# Binary Image Classifier: Open Mouth vs Closed Mouth

This notebook trains a binary image classifier using **MobileNetV2** with transfer learning to distinguish between **open mouth** and **closed mouth** images.

### Pipeline Overview
1. Install & import dependencies
2. Load and augment the dataset
3. Build the model (MobileNetV2 + custom head)
4. Compile the model
5. Train for 15 epochs
6. Plot accuracy & loss curves
7. Export the model to TensorFlow.js format

## 1. Install & Import Dependencies

We install `tensorflowjs` (needed later for export) and import TensorFlow / Keras, Matplotlib, and OS utilities.

In [2]:
!pip install tensorflowjs --quiet

import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


TensorFlow version: 2.16.1
GPU available: []


## 2. Load & Augment the Dataset

The dataset is stored under `training/dataset/` with two sub-folders:
- `open_mouth/`
- `closed_mouth/`

We use `ImageDataGenerator` to apply data augmentation on the training split:
- **Rescale** pixel values to [0, 1]
- **Rotation** up to 20°
- **Zoom** up to 20%
- **Horizontal flip**

A validation split of 20% is reserved for evaluation.

In [ ]:
# ----- Configuration -----
DATASET_DIR = os.path.join('training', 'dataset')
IMG_SIZE = (224, 224)
BATCH_SIZE = 4  # Small batch size for small datasets
NUM_CLASSES = 2
EPOCHS = 15

# ----- Data Augmentation (training) -----
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 80/20 train-val split
)

# ----- Training generator -----
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# ----- Validation generator (only rescaling, no augmentation) -----
val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"\nClass indices: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

Found 6 images belonging to 2 classes.
Found 0 images belonging to 2 classes.

Class indices: {'closed_mouth': 0, 'open_mouth': 1}
Training samples: 6
Validation samples: 0


## 3. Build the Model

We use **MobileNetV2** pre-trained on ImageNet as the feature extractor (base). The base layers are frozen so only the new classification head is trained.

**Architecture:**
- MobileNetV2 base (frozen)
- Global Average Pooling
- Dropout (0.3) for regularisation
- Dense layer with 128 units + ReLU
- Output Dense layer with 2 units + **softmax**

In [4]:
# ----- Load pre-trained MobileNetV2 (exclude top) -----
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the base model layers
base_model.trainable = False

# ----- Add custom classification head -----
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,210 (9.24 MB)

 Trainable params: 164,226 (641.51 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 4. Compile the Model

We compile with:
- **Optimizer:** Adam (learning rate 1e-3)
- **Loss:** Categorical Cross-Entropy (2 classes)
- **Metric:** Accuracy

In [5]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")

Model compiled successfully.


## 5. Train the Model

Train for **15 epochs** using the augmented training set and validate on the held-out 20% split.

In [7]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator
)

print("\nTraining complete.")

Epoch 1/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.5000 - loss: 0.5482

ValueError: The PyDataset has length 0

## 6. Plot Accuracy & Loss Curves

Visualise how accuracy and loss evolve over the 15 epochs for both training and validation sets. This helps diagnose overfitting or underfitting.

In [8]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy ---
ax1.plot(epochs_range, acc, label='Training Accuracy')
ax1.plot(epochs_range, val_acc, label='Validation Accuracy')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

# --- Loss ---
ax2.plot(epochs_range, loss, label='Training Loss')
ax2.plot(epochs_range, val_loss, label='Validation Loss')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

NameError: name 'history' is not defined

## 7. Export Model to TensorFlow.js

First we save the Keras model to a temporary directory, then convert it to **TensorFlow.js** format inside `model_export/`. This allows the model to be loaded directly in a browser or Node.js application.

In [9]:
import tensorflowjs as tfjs

EXPORT_DIR = 'model_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Export directly from the Keras model to TF.js Layers format
tfjs.converters.save_keras_model(model, EXPORT_DIR)

print(f"\nModel exported to '{EXPORT_DIR}/' successfully.")
print("Contents:")
for f in os.listdir(EXPORT_DIR):
    print(f"  {f}")

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflowjs\read_weights.py:28: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  np.uint8, np.uint16, np.object, np.bool]


AttributeError: module 'numpy' has no attribute 'object'.
`np.object` was a deprecated alias for the builtin `object`. To avoid this error in existing code, use `object` by itself. Doing this will not modify any behavior and is safe. 
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

## Done!

The trained model has been exported to `model_export/`. You can now load it in a web application using:

```js
const model = await tf.loadLayersModel('model_export/model.json');
```

**Class mapping:**
- `0` → `closed_mouth`
- `1` → `open_mouth`

*(Verify the exact indices printed in Step 2 — they depend on alphabetical folder order.)*